# Day2 모델링 (3) - 피처군별 영향력과 피처 축소

평가 기준: **사용한 피처가 종속변수에 어느 정도 영향이 있는지**, 그리고 **불필요한 컬럼이 많다고 좋은 것이 아니라는 점**을 보인다.

1. 피처군별 기여: 어떤 정보(수치4, RDKit, 핑거프린트, 추가 지문)를 넣으면 성능이 얼마나 오르는지
2. 피처 축소: 중요도 상위 k개만 써도 성능이 유지되는지(선택은 fold 안의 학습 데이터에서만 수행)

모든 비교는 학습 80%의 5-Fold 교차검증(LightGBM 고정)이고 홀드아웃은 열지 않는다.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data import load_all  # noqa: E402
from preprocess import build_pipeline  # noqa: E402

warnings.filterwarnings("ignore")
SEED = 42
TAB = ROOT / "reports/tables"
d = load_all(ROOT, seed=SEED, extra=True)
X_tr, y_tr, GROUPS = d.X_tr, d.y_tr, d.groups
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
THS = np.round(np.arange(0.30, 0.71, 0.01), 2)
print("학습", X_tr.shape, {k: len(v) for k, v in GROUPS.items()})


def lgbm():
    return LGBMClassifier(n_estimators=600, learning_rate=0.03, num_leaves=31, colsample_bytree=0.3, subsample=0.8,
                          subsample_freq=1, random_state=SEED, n_jobs=3, verbose=-1)


def score(pipe, label, n_features):
    proba = cross_val_predict(pipe, X_tr, y_tr, cv=CV, method="predict_proba", n_jobs=5)[:, 1]
    f1s = np.array([f1_score(y_tr, (proba >= t).astype(int)) for t in THS])
    return {"구성": label, "피처 수": n_features, "AUC": roc_auc_score(y_tr, proba),
            "F1@0.5": f1_score(y_tr, (proba >= 0.5).astype(int)), "최대 F1": f1s.max(), "최적 임계값": THS[f1s.argmax()]}

학습 (6679, 8577) {'cont': 4, 'rdkit': 214, 'fp': 3072, 'maccs': 167, 'morgan_cnt': 2048, 'ap_cnt': 2048, 'tt_cnt': 1024}


## 1. 피처군별 기여 (ablation)

같은 모델에 피처군을 바꿔 넣는다. 한 군만 쓴 경우와 누적해서 더한 경우를 모두 본다. "피처 수"는 원래 열 개수이며 상수열은 fold 안에서 제거된다.

In [2]:
SETS = [
    ("수치4(cont)만", ["cont"]),
    ("RDKit 파생만", ["rdkit"]),
    ("핑거프린트(fp)만", ["fp"]),
    ("MACCS만", ["maccs"]),
    ("Morgan 카운트만", ["morgan_cnt"]),
    ("AtomPair 카운트만", ["ap_cnt"]),
    ("TopTorsion 카운트만", ["tt_cnt"]),
    ("cont+rdkit", ["cont", "rdkit"]),
    ("cont+rdkit+fp", ["cont", "rdkit", "fp"]),
    ("cont+rdkit+fp+maccs", ["cont", "rdkit", "fp", "maccs"]),
    ("전체(+카운트 지문)", ["cont", "rdkit", "fp", "maccs", "morgan_cnt", "ap_cnt", "tt_cnt"]),
]
rows = []
for label, groups in SETS:
    n = sum(len(GROUPS[g]) for g in groups)
    row = score(build_pipeline(lgbm(), GROUPS, groups), label, n)
    rows.append(row)
    print(f"{label:<22} 피처 {n:>5} | AUC {row['AUC']:.4f} | 최대 F1 {row['최대 F1']:.4f} @ {row['최적 임계값']:.2f}", flush=True)
abl = pd.DataFrame(rows)
abl.round(4).to_csv(TAB / "modeling_feature_groups.csv", index=False)
abl.round(4)

수치4(cont)만             피처     4 | AUC 0.6733 | 최대 F1 0.7127 @ 0.30


RDKit 파생만              피처   214 | AUC 0.8693 | 최대 F1 0.8162 @ 0.35


핑거프린트(fp)만             피처  3072 | AUC 0.8729 | 최대 F1 0.8189 @ 0.44


MACCS만                 피처   167 | AUC 0.8481 | 최대 F1 0.8068 @ 0.38


Morgan 카운트만            피처  2048 | AUC 0.8690 | 최대 F1 0.8173 @ 0.39


...

cont+rdkit             피처   218 | AUC 0.8700 | 최대 F1 0.8161 @ 0.43


cont+rdkit+fp          피처  3290 | AUC 0.8817 | 최대 F1 0.8262 @ 0.35


cont+rdkit+fp+maccs    피처  3457 | AUC 0.8817 | 최대 F1 0.8261 @ 0.37


전체(+카운트 지문)            피처  8577 | AUC 0.8863 | 최대 F1 0.8295 @ 0.37


,구성,피처 수,AUC,F1@0.5,최대 F1,최적 임계값
0,수치4(cont)만,4,0.6733,0.6647,0.7127,0.30
1,RDKit 파생만,214,0.8693,0.8075,0.8162,0.35
2,핑거프린트(fp)만,3072,0.8729,0.8153,0.8189,0.44
3,MACCS만,167,0.8481,0.7929,0.8068,0.38
4,Morgan 카운트만,2048,0.8690,0.8100,0.8173,0.39
5,AtomPair 카운트만,2048,0.8732,0.8136,0.8227,0.38
6,TopTorsion 카운트만,1024,0.8577,0.7998,0.8090,0.44
7,cont+rdkit,218,0.8700,0.8115,0.8161,0.43
8,cont+rdkit+fp,3290,0.8817,0.8195,0.8262,0.35
9,cont+rdkit+fp+maccs,3457,0.8817,0.8193,0.8261,0.37


## 2. 중요도 기반 피처 축소 (fold 안에서 선택)

각 fold의 학습 데이터로 LightGBM을 학습해 gain 중요도 상위 k개만 남기고 최종 모델을 학습한다. 선택에 라벨을 쓰지만 `Pipeline` 안이라 검증 fold로 새지 않는다.

In [3]:
ALL = ["cont", "rdkit", "fp", "maccs", "morgan_cnt", "ap_cnt", "tt_cnt"]


def with_selection(k):
    base = build_pipeline(lgbm(), GROUPS, ALL)
    selector = SelectFromModel(
        LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, colsample_bytree=0.3, subsample=0.8,
                       subsample_freq=1, importance_type="gain", random_state=SEED, n_jobs=3, verbose=-1),
        threshold=-np.inf, max_features=k)
    return Pipeline([("prep", base.named_steps["prep"]), ("sel", selector), ("model", base.named_steps["model"])])


rows = []
for k in (30, 100, 300, 1000, 3000):
    row = score(with_selection(k), f"중요도 상위 {k}개", k)
    rows.append(row)
    print(f"상위 {k:>5}개 | AUC {row['AUC']:.4f} | F1@0.5 {row['F1@0.5']:.4f} | 최대 F1 {row['최대 F1']:.4f} @ {row['최적 임계값']:.2f}", flush=True)
rows.append({**abl.iloc[-1].to_dict(), "구성": "전체(선택 없음)"})
sel = pd.DataFrame(rows)
sel.round(4).to_csv(TAB / "modeling_feature_selection.csv", index=False)
sel.round(4)

상위    30개 | AUC 0.8548 | F1@0.5 0.8046 | 최대 F1 0.8082 @ 0.45


상위   100개 | AUC 0.8705 | F1@0.5 0.8118 | 최대 F1 0.8153 @ 0.36


상위   300개 | AUC 0.8780 | F1@0.5 0.8196 | 최대 F1 0.8226 @ 0.39


상위  1000개 | AUC 0.8827 | F1@0.5 0.8236 | 최대 F1 0.8265 @ 0.35


상위  3000개 | AUC 0.8856 | F1@0.5 0.8255 | 최대 F1 0.8302 @ 0.41


,구성,피처 수,AUC,F1@0.5,최대 F1,최적 임계값
0,중요도 상위 30개,30,0.8548,0.8046,0.8082,0.45
1,중요도 상위 100개,100,0.8705,0.8118,0.8153,0.36
2,중요도 상위 300개,300,0.8780,0.8196,0.8226,0.39
3,중요도 상위 1000개,1000,0.8827,0.8236,0.8265,0.35
4,중요도 상위 3000개,3000,0.8856,0.8255,0.8302,0.41
5,전체(선택 없음),8577,0.8863,0.8261,0.8295,0.37
